# Sunset : the four-loop interval certificate: worked example

Companion worksheet for [The three- and four-loop
equal-mass sunset integrals and local Calabi--Yau fourfolds and fivefolds](), Sections *The four-loop connection quantities to certify* (§9<!--`ext:cert:setup`-->) to *Execution and interpretation* (§13<!--`ext:cert:run`-->) and Appendix A<!--`ext:code`-->.

It runs the two programs listed in the appendix, `certify.py` and `validate.py`, which must sit in the same folder as this notebook, and checks every number the paper quotes against what they produce.

**Requirements** (from the appendix): Python ≥ 3.10 and `python-flint==0.9.0`. Do not run Python with `-O`, because the certificate depends on its `assert` statements.

| paper | code |
|---|---|
| operator $P_4$, $Q_1,Q_2,Q_3$ | `certify.Q` |
| germ recurrence (<!--`ext:cert:germrec`-->), majorant (<!--`ext:cert:germmajorant`-->) | `certify.germ_coeff` |
| initial tail $e_g$ (10.3<!--`ext:cert:initialtail`-->) | `certify.initial` |
| $M(c)$ (11.1<!--`ext:cert:M`-->), step tail (11.2<!--`ext:cert:steptail`-->) | `certify.step` |
| Frobenius recurrence at $y=\infty$, majorant (12.1<!--`ext:cert:frobmajorant`-->), tails (12.2<!--`ext:cert:frobjets`-->) | `certify.frob`, `certify.eval_frob` |
| $d_j$, $E_k$ (9.1<!--`ext:cert:Ek`-->), Theorem 9.1<!--`ext:cert:main`--> | `certify.run` |
| 50 square-free coefficients, $15497623/21870000$, $1807/405$, $871/450$, $13/1350$, Table 2<!--`ext:cert:intervals`--> | `validate.py` |


In [1]:
import sys, os, json, subprocess, flint
from fractions import Fraction as F
from decimal import Decimal, getcontext
getcontext().prec = 40
import certify, validate
print("python", sys.version.split()[0], "| python-flint", flint.__version__)

python 3.14.8 | python-flint 0.9.0


## 1. The operator $P_4$

$$P_4=\theta_y^4+yQ_1(\theta_y)+y^2Q_2(\theta_y)+y^3Q_3(\theta_y),$$
$$Q_1=35z^4+70z^3+63z^2+28z+5,\quad Q_2=259z^4+1036z^3+1580z^2+1088z+285,\quad Q_3=225(z+1)^2(z+2)^2.$$

`certify.Q[j][k]` is the coefficient of $y^j\theta_y^k$. The cell below checks two things exactly. First, $Q_3$ factors as stated. Second, the coefficient of $\theta_y^4$ is $p_4=(1+y)(1+9y)(1+25y)$, which is the denominator used in $M(c)$.

In [2]:
def pmul(a, b):
    c = [0]*(len(a)+len(b)-1)
    for i, x in enumerate(a):
        for j, z in enumerate(b): c[i+j] += x*z
    return c
Q = certify.Q
for j, row in enumerate(Q):
    print(f"y^{j}: " + " + ".join(f"{c}*th^{k}" for k, c in reversed(list(enumerate(row))) if c))
print("Q_3 == 225 (z+1)^2 (z+2)^2 :", Q[3] == [225*c for c in pmul(pmul([1,1],[1,1]), pmul([2,1],[2,1]))])
print("p_4 == (1+y)(1+9y)(1+25y)  :", [Q[j][4] for j in range(4)] == pmul(pmul([1,1],[1,9]),[1,25]))

y^0: 1*th^4
y^1: 35*th^4 + 70*th^3 + 63*th^2 + 28*th^1 + 5*th^0
y^2: 259*th^4 + 1036*th^3 + 1580*th^2 + 1088*th^1 + 285*th^0
y^3: 225*th^4 + 1350*th^3 + 2925*th^2 + 2700*th^1 + 900*th^0
Q_3 == 225 (z+1)^2 (z+2)^2 : True
p_4 == (1+y)(1+9y)(1+25y)  : True


`certify.ODE` rewrites $P_4$ in ordinary derivatives, $P_4=\sum_r a_r(y)\,y^r\partial_y^r$, using Stirling numbers of the second kind. The step routine needs this form.

The cell checks the conversion on monomials. On $y^s$, the operator $\theta_y^k$ acts as multiplication by $s^k$, while $y^r\partial_y^r$ acts as multiplication by the falling factorial $s(s-1)\cdots(s-r+1)$.

In [3]:
def falling(s, r):
    p = 1
    for i in range(r): p *= s - i
    return p
ok = all(sum(Q[j][k]*s**k for k in range(5)) ==
         sum(certify.ODE[r][j+r]*falling(s, r) for r in range(5) if j+r < len(certify.ODE[r]))
         for s in range(-6, 12) for j in range(4))
print("ODE table reproduces P_4 :", ok)

ODE table reproduces P_4 : True


**Equation at $y=\infty$ (Certificate III).** Put $x=1/y$, $D=x\partial_x$ and $W=xF$. The paper states
$$[225D^2(D-1)^2+xD^2(259D^2+26)+x^2Q_1(D)+x^3(D+1)^4]F=b\,x^2 .$$
Apply $x^2P_4$ to $W=x^{s+1}$. The term $y^jQ_j(\theta_y)$ contributes $Q_j(-s-1)\,x^{s+3-j}$, so the stated equation is equivalent to four polynomial identities in $s$. Each side has degree at most 4, so checking 20 integer values of $s$ proves them.

In [4]:
Qp = lambda j, s: sum(c*s**k for k, c in enumerate(Q[j]))
checks = {
 "Q_3(-s-1) = 225 s^2 (s-1)^2": all(Qp(3,-s-1) == 225*s**2*(s-1)**2 for s in range(-10,10)),
 "Q_2(-s-1) = s^2 (259 s^2 + 26)": all(Qp(2,-s-1) == s**2*(259*s**2+26) for s in range(-10,10)),
 "Q_1(-s-1) = Q_1(s)":             all(Qp(1,-s-1) == Qp(1,s) for s in range(-10,10)),
 "Q_0(-s-1) = (s+1)^4":            all(Qp(0,-s-1) == (s+1)**4 for s in range(-10,10)),
}
for k, v in checks.items(): print(f"{k:34s} {v}")

Q_3(-s-1) = 225 s^2 (s-1)^2        True
Q_2(-s-1) = s^2 (259 s^2 + 26)     True
Q_1(-s-1) = Q_1(s)                 True
Q_0(-s-1) = (s+1)^4                True


## 2. Certificate I: exact normalization and initial tails

The germ recurrence (10.1<!--`ext:cert:germrec`-->) works in $\mathbb Q[\varepsilon]/(\varepsilon^5)$:
$$u_D(\varepsilon)=-(D+\varepsilon)^{-4}\sum_{l=1}^{\min(3,D)}Q_l(D-l+\varepsilon)\,u_{D-l}(\varepsilon).$$
It must agree exactly with the square-free ambient formula on the $10\times5=50$ coefficients for $D=0,\dots,9$.

In [18]:
a = validate.exact_recurrence(9); b = validate.squarefree(9)
print("50 coefficients agree exactly:", a == b, "  (", sum(len(r) for r in a), "coefficients )")
for D in range(4):
    print(f"u_{D}(eps) =", " + ".join(f"({c})eps^{k}" for k, c in enumerate(a[D]) if c))

50 coefficients agree exactly: True   ( 50 coefficients )
u_0(eps) = (1)eps^0
u_1(eps) = (-5)eps^0 + (-8)eps^1 + (-1)eps^2 + (2)eps^3
u_2(eps) = (45)eps^0 + (100)eps^1 + (197/4)eps^2 + (-89/4)eps^3 + (-61/4)eps^4
u_3(eps) = (-545)eps^0 + (-4148/3)eps^1 + (-33637/36)eps^2 + (19295/108)eps^3 + (2818/9)eps^4


**Germ majorant** (10.2<!--`ext:cert:germmajorant`-->): $\|u_D\|\le 2\cdot100^D$. Ten base cases are checked directly, and the induction step needs
$$\sum_{l=1}^3\frac{Q_l(10)}{9^4\,100^l}=\frac{15497623}{21870000}<1.$$

In [19]:
print("base cases D=0..9:", all(sum(map(abs, r)) <= 2*100**D for D, r in enumerate(a)))
gf = sum(F(sum(Q[l][k]*10**k for k in range(5)), 9**4*100**l) for l in range(1, 4))
print("induction factor =", gf, " == 15497623/21870000:", gf == F(15497623, 21870000), " < 1:", gf < 1)

base cases D=0..9: True
induction factor = 15497623/21870000  == 15497623/21870000: True  < 1: True


**Initial tail** (10.3<!--`ext:cert:initialtail`-->): $e_g=1024\,q_0^{N_g+1}(N_g+2)^3\,\dfrac{1+4q_0+q_0^2}{(1-q_0)^4}$ with $q_0=100/512$ and $N_g=450$. The paper claims $e_g<5.4\times10^{-309}$.

In [20]:
from flint import arb, ctx
ctx.prec = 1400
q0 = arb(100)/512; Ng = 450
eg = 1024*q0**(Ng+1)*(Ng+2)**3*(1+4*q0+q0*q0)/(1-q0)**4
print("e_g =", eg.str(10), "  < 5.4e-309:", eg < arb("5.4e-309"))

e_g = [5.399845183e-309 +/- 1.99e-319]   < 5.4e-309: True


## 3. Certificate II: the continuation bound $M(c)$

$$M(c)=\frac1{1-r}\max\Big\{1,\frac{1+\sum_{a=0}^3p_a(c(1+r))}{(1+c(1-r))(1+9c(1-r))(1+25c(1-r))}\Big\},\qquad r=\tfrac14 .$$
Here $\sum_{a\le3}p_a(y)=\sum_l\big(\sum_{a\le3}[Q_l]_a\big)y^l$, which is exactly `numer` in `certify.step`. The table shows how large the propagation factor $e^{M(c)r}$ gets along the path from $y_0=1/512$ to $y=32$.

In [21]:
r = F(1, 4)
def M(c):
    c = F(c); ymax = c*(1+r)
    num = 1 + sum(sum(Q[l][:4])*ymax**l for l in range(1, 4))
    den = (1+c*(1-r))*(1+9*c*(1-r))*(1+25*c*(1-r))
    return max(F(1), num/den)/(1-r)
import math
print(f"{'c':>10} {'M(c)':>10} {'exp(M r)':>10}")
for c in (F(1,512), F(1,64), F(1,8), 1, 4, 32):
    m = M(c); print(f"{str(c):>10} {float(m):10.4f} {math.exp(float(m*r)):10.4f}")

         c       M(c)   exp(M r)
     1/512     1.8117     1.5729
      1/64     5.3686     3.8272
       1/8    30.5236  2060.9241
         1   108.6250 622030058099.4927
         4   169.9453 2828655035341327872.0000
        32   208.7209 45875395095690525802496.0000


## 4. Certificate III: Frobenius majorant at $y=\infty$

For $k\ge4$: $\|P_k\|\le5\|P_{k-1}\|+2\|P_{k-2}\|+\tfrac1{100}\|P_{k-3}\|$, with dominating constants $1807/405$, $871/450$, $13/1350$. Together with $5/8+2/8^2+1/(100\cdot8^3)<1$ this gives $\|P_k\|\le2\cdot8^k$ (12.1<!--`ext:cert:frobmajorant`-->).

In [22]:
H1 = (F(259)+F(26,16))*F(16,9)*F(13,6)/225; H2 = F(201,225)*F(13,6); H3 = F(1,225)*F(13,6)
print("H1 =", H1, "== 1807/405:", H1 == F(1807,405), " < 5:", H1 < 5)
print("H2 =", H2, "== 871/450 :", H2 == F(871,450),  " < 2:", H2 < 2)
print("H3 =", H3, "== 13/1350 :", H3 == F(13,1350),  " < 1/100:", H3 < F(1,100))
print("5/8 + 2/64 + 1/51200 =", F(5,8)+F(2,64)+F(1,51200), "< 1")

H1 = 1807/405 == 1807/405: True  < 5: True
H2 = 871/450 == 871/450 : True  < 2: True
H3 = 13/1350 == 13/1350 : True  < 1/100: True
5/8 + 2/64 + 1/51200 = 33601/51200 < 1


## 5. Running the certificate (Appendix A<!--`ext:code`-->)

The paper's commands are
```
python3 certify.py
python3 certify.py --bits 1500 --order 420 --match 48
python3 validate.py
```
**First run:** 1400 bits, $N=400$, $N_g=450$, $N_f=250$, matching at $y=32$. The paper states it takes 161 steps. Each run takes about 6–10 s.

In [23]:
c32 = certify.run()                                  # writes certificate_32.json

step 40 y [0.022074333 +/- 9.41e-11] radius [2.5930e-209 +/- 1.42e-214]
step 80 y [0.24948540 +/- 7.02e-10] radius [9.0816e-195 +/- 1.23e-200]
step 120 y [2.8196986 +/- 1.94e-8] radius [4.2752e-174 +/- 3.39e-179]
step 160 y [31.868398 +/- 1.64e-7] radius [1.8742e-149 +/- 2.39e-155]
R [-1.088825123504371170391756125490055353332 +/- 1.06e-40] E ['[1.95921991624736653018094754405 +/- 1.28e-30]', '[-0.535347537401972845058209420122 +/- 1.19e-31]', '[0.420379682614378171767897648547 +/- 8.12e-33]', '[0.748770280780437029430306548905 +/- 1.34e-31]'] seconds 4.044053077697754


**Second, independent run:** 1500 bits, $N=420$, matching at $y=48$.

In [24]:
c48 = certify.run(bits=1500, N=420, match=48)        # writes certificate_48.json

step 40 y [0.022074333 +/- 9.41e-11] radius [2.7291e-221 +/- 2.68e-226]
step 80 y [0.24948540 +/- 7.02e-10] radius [9.5578e-207 +/- 2.74e-212]
step 120 y [2.8196986 +/- 1.94e-8] radius [4.4993e-186 +/- 1.85e-191]
step 160 y [31.868398 +/- 1.64e-7] radius [1.9725e-161 +/- 3.09e-166]
R [-1.088825123504371170391756125490055353332 +/- 1.06e-40] E ['[1.95921991624736653018094754405 +/- 1.28e-30]', '[-0.535347537401972845058209420122 +/- 1.19e-31]', '[0.420379682614378171767897648547 +/- 8.12e-33]', '[0.748770280780437029430306548905 +/- 1.34e-31]'] seconds 4.444267988204956


## 6. Results compared with the paper

`box()` exports every ball as an outward-rounded rational interval with denominator $10^{24}$. The cell below compares these intervals with the paper's claims from §13<!--`ext:cert:run`-->, Theorem 9.1<!--`ext:cert:main`--> and Table 2<!--`ext:cert:intervals`-->.

One caveat about the tail-bound rows: a bound smaller than $10^{-24}$ exports as the useless interval $[0,10^{-24}]$. So the ball strings are used for $e_g$ and the step remainders $e$, and the rational intervals are used for everything else.

In [25]:
def iv(b): return (Decimal(int(b['outer_lower_numerator']))/Decimal(int(b['outer_denominator'])),
                   Decimal(int(b['outer_upper_numerator']))/Decimal(int(b['outer_denominator'])))
def ballval(b): return float(b['ball'].strip('[').split(' ')[0])
def ballrad(b): return float(b['radius'].strip('[').split(' ')[0])
print(f"{'claim':52s} {'run y=32':>14} {'run y=48':>14}")
rows = [("steps (paper: 161 in first run)", lambda c: c['steps']),
        ("initial tail e_g < 5.4e-309", lambda c: f"{ballval(c['initial_tail_bound']):.4e}"),
        ("largest step remainder e < 1.25e-218 (first run)", lambda c: f"{ballval(c['largest_step_tail_W_bound']):.4e}"),
        ("max radius of the E_k balls < 6e-139 (first run)", lambda c: f"{max(ballrad(e) for e in c['E']):.3e}"),
        ("b_0 < 0", lambda c: iv(c['connections'][0][1])[1] < 0)]
for name, f in rows: print(f"{name:52s} {str(f(c32)):>14} {str(f(c48)):>14}")

claim                                                      run y=32       run y=48
steps (paper: 161 in first run)                                 161            167
initial tail e_g < 5.4e-309                             5.3998e-309    5.3998e-309
largest step remainder e < 1.25e-218 (first run)        1.2463e-218    1.9415e-230
max radius of the E_k balls < 6e-139 (first run)         5.796e-139     3.762e-147
b_0 < 0                                                        True           True


**Endpoint $Q_\infty=e^{R_\infty}$.** Theorem 9.1<!--`ext:cert:main`--> encloses it as
$0.336611738699193750820017\le Q_\infty\le0.336611738699193750820018$.

In [26]:
paperQ = (Decimal("0.336611738699193750820017"), Decimal("0.336611738699193750820018"))
for name, c in (("y=32", c32), ("y=48", c48)):
    print(name, "R_inf in", [str(v) for v in iv(c['R_infinity'])])
    print(name, "Q_inf in", [str(v) for v in iv(c['Q_infinity'])], " == paper:", iv(c['Q_infinity']) == paperQ)

y=32 R_inf in ['-1.088825123504371170391757', '-1.088825123504371170391756']
y=32 Q_inf in ['0.336611738699193750820017', '0.336611738699193750820018']  == paper: True
y=48 R_inf in ['-1.088825123504371170391757', '-1.088825123504371170391756']
y=48 Q_inf in ['0.336611738699193750820017', '0.336611738699193750820018']  == paper: True


**Endpoint expansion** $W_j=x(a_j+b_j\log x)+O(x^2|\log x|)$ and the limits $C_j(\infty)$.

The connection rows are $(a_j,b_j,c_j,e_j)$, where $P_{j,1}(t)=c_j+e_jt$. The row $j=4$ is the inhomogeneous one, and the paper stresses that it must be included. $R_\infty=C_1(\infty)$ is the $C$-limit of row 0.

In [27]:
print(f"{'j':>2} {'a_j':>26} {'b_j':>26} {'C_(j+1)(inf)':>26}")
for j, (row, Cl) in enumerate(zip(c32['connections'], c32['C_limits'])):
    print(f"{j:>2} {row[0]['ball'][:26]:>26} {row[1]['ball'][:26]:>26} {Cl['ball'][:26]:>26}")

 j                        a_j                        b_j               C_(j+1)(inf)
 0 [0.24751276401479937725976 [-0.1649669005300320295198 [-1.0888251235043711703917
 1 [-0.5239778351045394122092 [0.13381384748223276787065 [0.12705662628037002422643
 2 [0.44022991267911490522484            [+/- 1.06e-141] [0.80137126877306285693315
 3 [-0.1587454007378684139759 [-0.1321997343833114230001 [-0.8725531706946847890363
 4 [-0.0870208824267187493358 [0.10723457273624199589148 [0.51659063186567297886066


**The four signs and Table 2<!--`ext:cert:intervals`-->.**
$$d_j=\frac{a_jb_0-a_0b_j}{b_0^2},\qquad E_k=\sum_{h=1}^{k-1}\frac{(-R_\infty)^{k-1-h}}{(k-1-h)!}\,d_h .$$
Theorem `ext:cert:main` states $E_2>0$, $E_3<0$, $E_4>0$ and $E_5>0$.

In [28]:
paper_table = [("1.959219916247366530180947", "1.959219916247366530180948"),
               ("-0.535347537401972845058210", "-0.535347537401972845058209"),
               ("0.420379682614378171767897", "0.420379682614378171767898"),
               ("0.748770280780437029430306", "0.748770280780437029430307")]
signs = ['+', '-', '+', '+']
for k, (e32, e48, (lo, hi), s) in enumerate(zip(c32['E'], c48['E'], paper_table, signs), start=2):
    i32, i48 = iv(e32), iv(e48)
    same = i32 == (Decimal(lo), Decimal(hi))
    sign_ok = i32[0] > 0 if s == '+' else i32[1] < 0
    overlap = max(i32[0], i48[0]) <= min(i32[1], i48[1])
    print(f"E_{k} in [{lo}, {hi}]   table: {same}   sign {s}: {sign_ok}   runs overlap: {overlap}")

E_2 in [1.959219916247366530180947, 1.959219916247366530180948]   table: True   sign +: True   runs overlap: True
E_3 in [-0.535347537401972845058210, -0.535347537401972845058209]   table: True   sign -: True   runs overlap: True
E_4 in [0.420379682614378171767897, 0.420379682614378171767898]   table: True   sign +: True   runs overlap: True
E_5 in [0.748770280780437029430306, 0.748770280780437029430307]   table: True   sign +: True   runs overlap: True


## 7. `validate.py`

This script redoes the exact rational checks independently of the ball arithmetic. It also reads both certificates and checks that the signs are proved by their rational intervals and that the two runs overlap. It writes `validation.json` and `exact_germ_check.json`. It is run here exactly as the paper says.

In [16]:
res = subprocess.run([sys.executable, "validate.py"], capture_output=True, text=True)
print("exit code:", res.returncode)
v = json.loads(res.stdout)
for k in ('exact_squarefree_normalization_coefficients', 'germ_base_cases', 'germ_induction_factor',
          'frobenius_H1_bound', 'frobenius_H2_bound', 'frobenius_H3_bound', 'two_certificates_pass', 'status'):
    print(f"{k:45s} {v[k]}")

exit code: 0
exact_squarefree_normalization_coefficients   50
germ_base_cases                               10
germ_induction_factor                         15497623/21870000
frobenius_H1_bound                            1807/405
frobenius_H2_bound                            871/450
frobenius_H3_bound                            13/1350
two_certificates_pass                         True
status                                        All assertions passed. The analytic tail proofs in the report are part of the certificate.


## 8. What the certificate proves

The signs of $E_k$ give $\partial_Q^2K_k\sim E_k\big/\big(Q_\infty^2\,\Delta\,|\log\Delta|^2\big)$ with $\Delta=R_\infty-R\downarrow0$. So each diagonal descendant potential $-20K_2,-60K_3,-120K_4,-120K_5$ is $C^1$ up to $Q_\infty$ but has an unbounded second derivative there. Their individual $Q$-Taylor radii therefore satisfy
$$\tfrac1{20}\le\mathcal R_k\le Q_\infty\approx0.33661 .$$
The lower bound comes from Corollary 11.5<!--`cor:Qradius`--> of [Equal-mass sunset
integrals and genus-zero local\\ Gromov--Witten theory of Calabi--Yau
$n$-folds]() and is not computed here.

In [17]:
lo, hi = iv(c32['Q_infinity'])
print(f"1/20 = 0.05  <=  R_k  <=  Q_inf in [{lo}, {hi}]   (k = 2,3,4,5)")

1/20 = 0.05  <=  R_k  <=  Q_inf in [0.336611738699193750820017, 0.336611738699193750820018]   (k = 2,3,4,5)
